In [ ]:
!pip install torch torchvision pycocotools transformers ultralytics opencv-python pillow trl supervision albumentations peft
!git clone https://github.com/zjx0101/ObjectClear

In [ ]:
# Install required packages
!pip install ultralytics pycocotools matplotlib opencv-python pillow

import os
import shutil
import yaml
import json
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import cv2
from PIL import Image
from ultralytics import YOLO
import torch

print("Setting up directories...")
# Set up paths
coco_root = "/kaggle/input/coco-2017-dataset/coco2017"
working_dir = "/kaggle/working/coco-person-seg-optimized"
os.makedirs(working_dir, exist_ok=True)

# Create directory structure
for split in ['train', 'val']:
    os.makedirs(f"{working_dir}/images/{split}2017", exist_ok=True)
    os.makedirs(f"{working_dir}/labels/{split}2017", exist_ok=True)

print("Copying annotation files...")
# Copy annotations
annotations_src = f"{coco_root}/annotations"
annotations_dst = f"{working_dir}/annotations"
if os.path.exists(annotations_src):
    shutil.copytree(annotations_src, annotations_dst, dirs_exist_ok=True)

def convert_coco_to_yolo_person_only(coco_json_path, images_source_dir, images_dest_dir, labels_dest_dir, max_images=None):
    """Convert COCO annotations to YOLO format for person class only"""
    
    print(f"Processing {coco_json_path}...")
    
    # Load COCO annotations
    with open(coco_json_path, 'r') as f:
        coco_data = json.load(f)
    
    # Create mappings
    image_id_to_info = {img['id']: img for img in coco_data['images']}
    
    # Filter only person annotations (category_id = 1 in COCO)
    person_annotations = [ann for ann in coco_data['annotations'] if ann['category_id'] == 1]
    print(f"Found {len(person_annotations)} person annotations")
    
    # Group annotations by image_id
    annotations_by_image = {}
    for ann in person_annotations:
        image_id = ann['image_id']
        if image_id not in annotations_by_image:
            annotations_by_image[image_id] = []
        annotations_by_image[image_id].append(ann)
    
    print(f"Found {len(annotations_by_image)} images with person annotations")
    
    # Process images (limit to max_images if specified)
    image_ids = list(annotations_by_image.keys())
    if max_images:
        image_ids = image_ids[:max_images]
        print(f"Limiting to {max_images} images")
    
    processed_count = 0
    for image_id in image_ids:
        image_info = image_id_to_info.get(image_id)
        if not image_info:
            continue
            
        file_name = image_info['file_name']
        img_width = image_info['width']
        img_height = image_info['height']
        
        # Source and destination paths
        src_image_path = f"{images_source_dir}/{file_name}"
        dst_image_path = f"{images_dest_dir}/{file_name}"
        label_file_path = f"{labels_dest_dir}/{Path(file_name).stem}.txt"
        
        # Skip if source image doesn't exist
        if not os.path.exists(src_image_path):
            continue
        
        # Copy image
        shutil.copy2(src_image_path, dst_image_path)
        
        # Create label file
        with open(label_file_path, 'w') as label_file:
            for ann in annotations_by_image[image_id]:
                if 'segmentation' in ann and ann['segmentation']:
                    # COCO segmentation can be polygon or RLE, we use polygon
                    for segmentation in ann['segmentation']:
                        if isinstance(segmentation, list) and len(segmentation) >= 6:  # Need at least 3 points
                            # Convert polygon to YOLO format (normalized coordinates)
                            normalized_seg = []
                            for i in range(0, len(segmentation), 2):
                                x = segmentation[i] / img_width
                                y = segmentation[i+1] / img_height
                                # Ensure coordinates are within [0,1]
                                x = max(0, min(1, x))
                                y = max(0, min(1, y))
                                normalized_seg.extend([x, y])
                            
                            # Write to label file: class_id x1 y1 x2 y2 ...
                            if len(normalized_seg) >= 6:  # At least 3 points
                                line = "0 " + " ".join([f"{coord:.6f}" for coord in normalized_seg]) + "\n"
                                label_file.write(line)
        
        processed_count += 1
        if processed_count % 100 == 0:
            print(f"Processed {processed_count} images...")
    
    print(f"Completed processing {processed_count} images")
    return processed_count

print("\nConverting training data...")
# Convert training data - INCREASED SIGNIFICANTLY
train_count = convert_coco_to_yolo_person_only(
    coco_json_path=f"{working_dir}/annotations/instances_train2017.json",
    images_source_dir=f"{coco_root}/train2017",
    images_dest_dir=f"{working_dir}/images/train2017",
    labels_dest_dir=f"{working_dir}/labels/train2017",
    max_images=5000  # DRAMATICALLY INCREASED for better training
)

print("\nConverting validation data...")
# Convert validation data
val_count = convert_coco_to_yolo_person_only(
    coco_json_path=f"{working_dir}/annotations/instances_val2017.json",
    images_source_dir=f"{coco_root}/val2017",
    images_dest_dir=f"{working_dir}/images/val2017", 
    labels_dest_dir=f"{working_dir}/labels/val2017",
    max_images=500  # INCREASED for better validation
)

print(f"\nConversion completed!")
print(f"Training images: {train_count}")
print(f"Validation images: {val_count}")

# Create dataset YAML file
print("\nCreating dataset YAML file...")
dataset_yaml = {
    'path': working_dir,
    'train': 'images/train2017',
    'val': 'images/val2017',
    'nc': 1,
    'names': ['person']
}

yaml_path = f'{working_dir}/coco_person_seg.yaml'
with open(yaml_path, 'w') as f:
    yaml.dump(dataset_yaml, f)

print(f"Dataset YAML created at: {yaml_path}")

# Verify the dataset
def verify_dataset():
    print("\nVerifying dataset...")
    
    # Check training data
    train_images = os.listdir(f"{working_dir}/images/train2017")
    train_labels = os.listdir(f"{working_dir}/labels/train2017")
    print(f"Training images: {len(train_images)}, labels: {len(train_labels)}")
    
    # Check validation data  
    val_images = os.listdir(f"{working_dir}/images/val2017")
    val_labels = os.listdir(f"{working_dir}/labels/val2017")
    print(f"Validation images: {len(val_images)}, labels: {len(val_labels)}")

verify_dataset()

# ============================================================================
# OPTIMIZED TRAINING STRATEGY - NO FREEZING, MORE EPOCHS, BETTER PARAMS
# ============================================================================

print("\n" + "="*70)
print("OPTIMIZED TRAINING STRATEGY - FOCUS ON PERFORMANCE")
print("="*70)

# Use larger model for better performance
print("Loading YOLOv8s-seg model (larger than nano for better accuracy)...")
model = YOLO('yolov8s-seg.pt')  # UPGRADED to 's' model

print("\nStarting OPTIMIZED training (NO freezing, more epochs)...")

# OPTIMIZED TRAINING CONFIGURATION - KEY CHANGES:
results = model.train(
    data=yaml_path,
    epochs=50,                    # SIGNIFICANTLY INCREASED epochs
    imgsz=640,                   
    batch=8,                     
    patience=15,                  # Increased patience for convergence
    save=True,
    verbose=True,
    workers=4,
    single_cls=True,
    name='yolov8s_seg_person_optimized',
    project=working_dir,
    
    # OPTIMIZED HYPERPARAMETERS:
    lr0=0.01,                     # Higher initial LR for faster convergence
    lrf=0.1,                      # Final LR factor
    momentum=0.937,
    weight_decay=0.0005,
    warmup_epochs=3.0,            # Warmup for stable training
    warmup_momentum=0.8,
    warmup_bias_lr=0.1,
    
    # ENHANCED DATA AUGMENTATION:
    hsv_h=0.015,                  
    hsv_s=0.7,
    hsv_v=0.4,
    degrees=10.0,                 # Increased rotation
    translate=0.1, 
    scale=0.5,                    
    shear=2.0,                    
    perspective=0.0005,           # Increased perspective
    flipud=0.0,
    fliplr=0.5,
    mosaic=1.0,                   # Keep mosaic
    mixup=0.1,                    # Add mixup for regularization
    copy_paste=0.1,               # Add copy-paste augmentation
    
    # OPTIMIZATION:
    optimizer='SGD',              # SGD often works better for detection
    cos_lr=True,                  
    close_mosaic=10,              
    
    # REGULARIZATION:
    dropout=0.1,                  # Add dropout for regularization
    amp=True,                     # Mixed precision training
)

print("Optimized training completed!")

# Evaluate the optimized model
print("\nEvaluating optimized model...")
metrics_optimized = model.val(
    data=yaml_path,
    split='val',
    save_json=True,
    save_hybrid=True
)

print("\n=== OPTIMIZED MODEL Results ===")
print(f"mAP50: {metrics_optimized.box.map50:.4f}")
print(f"mAP50-95: {metrics_optimized.box.map:.4f}")
print(f"mAP50-seg: {metrics_optimized.seg.map50:.4f}")
print(f"mAP50-95-seg: {metrics_optimized.seg.map:.4f}")

# Compare with original model
print("\n" + "="*60)
print("COMPARISON WITH ORIGINAL MODEL")
print("="*60)

# Load original model for comparison
original_model = YOLO('yolov8s-seg.pt')
metrics_original = original_model.val(
    data=yaml_path,
    split='val',
    save_json=False,
    verbose=False
)

print(f"{'Metric':<15} {'Original':<10} {'Optimized':<10} {'Change':<10}")
print("-" * 50)
print(f"{'Det mAP50':<15} {metrics_original.box.map50:.4f}    {metrics_optimized.box.map50:.4f}    {metrics_optimized.box.map50 - metrics_original.box.map50:+.4f}")
print(f"{'Det mAP50-95':<15} {metrics_original.box.map:.4f}    {metrics_optimized.box.map:.4f}    {metrics_optimized.box.map - metrics_original.box.map:+.4f}")
print(f"{'Seg mAP50':<15} {metrics_original.seg.map50:.4f}    {metrics_optimized.seg.map50:.4f}    {metrics_optimized.seg.map50 - metrics_original.seg.map50:+.4f}")
print(f"{'Seg mAP50-95':<15} {metrics_original.seg.map:.4f}    {metrics_optimized.seg.map:.4f}    {metrics_optimized.seg.map - metrics_original.seg.map:+.4f}")

# ============================================================================
# ADVANCED STRATEGY: PROGRESSIVE TRAINING
# ============================================================================

def progressive_training():
    """Progressive training: start small, then increase complexity"""
    print("\n" + "="*60)
    print("ADVANCED: PROGRESSIVE TRAINING STRATEGY")
    print("="*60)
    
    # Stage 1: Quick training with smaller images
    print("\n--- STAGE 1: Quick training (320px) ---")
    model_stage1 = YOLO('yolov8s-seg.pt')
    
    results_stage1 = model_stage1.train(
        data=yaml_path,
        epochs=20,
        imgsz=320,  # Smaller for faster iterations
        batch=16,    # Larger batch for stability
        lr0=0.01,
        single_cls=True,
        name='stage1_quick',
        project=working_dir,
        exist_ok=True
    )
    
    # Stage 2: Refine with larger images
    print("\n--- STAGE 2: Refinement training (640px) ---")
    model_stage2 = YOLO(f'{working_dir}/stage1_quick/weights/best.pt')
    
    results_stage2 = model_stage2.train(
        data=yaml_path,
        epochs=30,
        imgsz=640,  # Full size
        batch=8,
        lr0=0.001,  # Lower LR for refinement
        single_cls=True,
        name='stage2_refined',
        project=working_dir,
        exist_ok=True
    )
    
    return model_stage2

# Uncomment to try progressive training
print("\nWould you like to try progressive training? It often gives better results.")
# progressive_model = progressive_training()

# ============================================================================
# ENSEMBLE STRATEGY
# ============================================================================

def create_ensemble_predictions(models_list, image_path):
    """Create ensemble predictions from multiple models"""
    all_predictions = []
    for model_path in models_list:
        model = YOLO(model_path)
        results = model.predict(image_path, conf=0.3)
        all_predictions.append(results[0])
    return all_predictions

# ============================================================================
# COMPREHENSIVE VISUALIZATION AND ANALYSIS
# ============================================================================

print("\nVisualizing optimized predictions...")
val_images_dir = f"{working_dir}/images/val2017"
val_labels_dir = f"{working_dir}/labels/val2017"

if os.path.exists(val_images_dir):
    val_images = os.listdir(val_images_dir)[:6]  # More samples
    
    for img_name in val_images:
        img_path = f"{val_images_dir}/{img_name}"
        label_path = f"{val_labels_dir}/{Path(img_name).stem}.txt"
        
        print(f"\nProcessing: {img_name}")
        
        # Run prediction with optimized model
        results = model.predict(img_path, conf=0.25)
        
        # Create comprehensive visualization
        fig, axes = plt.subplots(2, 2, figsize=(15, 12))
        axes = axes.ravel()
        
        # Original image
        orig_img = cv2.imread(img_path)
        orig_img = cv2.cvtColor(orig_img, cv2.COLOR_BGR2RGB)
        axes[0].imshow(orig_img)
        axes[0].set_title('Original Image')
        axes[0].axis('off')
        
        # Ground truth
        gt_img = orig_img.copy()
        if os.path.exists(label_path):
            with open(label_path, 'r') as f:
                lines = f.readlines()
            
            for line in lines:
                data = line.strip().split()
                if len(data) > 1:
                    segments = np.array([float(x) for x in data[1:]]).reshape(-1, 2)
                    segments[:, 0] *= orig_img.shape[1]
                    segments[:, 1] *= orig_img.shape[0]
                    
                    pts = segments.reshape((-1, 1, 2)).astype(np.int32)
                    cv2.polylines(gt_img, [pts], True, (0, 255, 0), 3)
                    mask = np.zeros_like(gt_img)
                    cv2.fillPoly(mask, [pts], (0, 255, 0))
                    gt_img = cv2.addWeighted(gt_img, 1, mask, 0.2, 0)
        
        axes[1].imshow(gt_img)
        axes[1].set_title('Ground Truth (Green)')
        axes[1].axis('off')
        
        # Optimized model prediction
        pred_img = results[0].plot()
        pred_img = cv2.cvtColor(pred_img, cv2.COLOR_BGR2RGB)
        axes[2].imshow(pred_img)
        axes[2].set_title('Optimized Model Prediction')
        axes[2].axis('off')
        
        # Confidence histogram
        if len(results[0].boxes) > 0:
            confidences = results[0].boxes.conf.cpu().numpy()
            axes[3].hist(confidences, bins=20, alpha=0.7, color='blue')
            axes[3].set_xlabel('Confidence')
            axes[3].set_ylabel('Count')
            axes[3].set_title(f'Detection Confidences\nMean: {np.mean(confidences):.3f}')
            axes[3].grid(True, alpha=0.3)
        else:
            axes[3].text(0.5, 0.5, 'No Detections', ha='center', va='center', transform=axes[3].transAxes)
            axes[3].set_title('Detection Confidences')
        
        plt.tight_layout()
        plt.show()

# ============================================================================
# PERFORMANCE ANALYSIS
# ============================================================================

def analyze_performance_improvement(original_metrics, optimized_metrics):
    """Analyze the performance improvement"""
    print("\n" + "="*60)
    print("PERFORMANCE IMPROVEMENT ANALYSIS")
    print("="*60)
    
    improvements = {
        'det_map50': optimized_metrics.box.map50 - original_metrics.box.map50,
        'det_map': optimized_metrics.box.map - original_metrics.box.map,
        'seg_map50': optimized_metrics.seg.map50 - original_metrics.seg.map50,
        'seg_map': optimized_metrics.seg.map - original_metrics.seg.map
    }
    
    print("Improvement Analysis:")
    for metric, improvement in improvements.items():
        if improvement > 0:
            print(f"✅ {metric}: +{improvement:.4f} (IMPROVED)")
        else:
            print(f"❌ {metric}: {improvement:.4f} (DECREASED)")
    
    avg_improvement = np.mean(list(improvements.values()))
    print(f"\nAverage Improvement: {avg_improvement:+.4f}")
    
    if avg_improvement > 0:
        print("🎉 Overall training was SUCCESSFUL!")
    else:
        print("⚠️  Training needs adjustment. Consider:")
        print("   - More training data")
        print("   - Different model architecture")
        print("   - Hyperparameter tuning")
        print("   - Longer training time")

# Run performance analysis
analyze_performance_improvement(metrics_original, metrics_optimized)

# ============================================================================
# FINAL RECOMMENDATIONS
# ============================================================================

print("\n" + "="*70)
print("FINAL RECOMMENDATIONS FOR MAXIMUM PERFORMANCE")
print("="*70)

print("""
🎯 KEY CHANGES MADE IN THIS VERSION:

1. MODEL SIZE: Upgraded to yolov8s-seg.pt (more capacity)
2. EPOCHS: Increased to 50 (proper convergence)
3. DATA: Increased to 5000 training images (reduces overfitting)
4. NO FREEZING: All layers trainable (better adaptation)
5. OPTIMIZED HYPERPARAMETERS: Better learning rate, augmentation
6. PROGRESSIVE TRAINING: Optional multi-stage approach

📊 EXPECTED RESULTS:
- mAP50-seg: Should improve by +0.05 to +0.15
- Better generalization on validation set
- More stable training curves

🚀 NEXT LEVEL OPTIMIZATIONS (if needed):
1. Use yolov8m-seg.pt (even larger model)
2. Train for 100+ epochs with early stopping
3. Use full COCO dataset (remove max_images limit)
4. Implement learning rate finder
5. Use test-time augmentation
""")

print(f"\nOptimized model saved at: {working_dir}")
print("🎉 OPTIMIZED training completed! Check the performance improvement above.")

In [ ]:
# Install required packages
!pip install ultralytics pycocotools matplotlib opencv-python pillow

import os
import shutil
import yaml
import json
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import cv2
from PIL import Image
from ultralytics import YOLO
import torch

print("Setting up directories...")
# Set up paths
coco_root = "/kaggle/input/coco-2017-dataset/coco2017"
working_dir = "/kaggle/working/coco-person-original-yolo"
os.makedirs(working_dir, exist_ok=True)

# Create directory structure
for split in ['train', 'val']:
    os.makedirs(f"{working_dir}/images/{split}2017", exist_ok=True)
    os.makedirs(f"{working_dir}/labels/{split}2017", exist_ok=True)

print("Copying annotation files...")
# Copy annotations
annotations_src = f"{coco_root}/annotations"
annotations_dst = f"{working_dir}/annotations"
if os.path.exists(annotations_src):
    shutil.copytree(annotations_src, annotations_dst, dirs_exist_ok=True)

def convert_coco_to_yolo_person_only(coco_json_path, images_source_dir, images_dest_dir, labels_dest_dir, max_images=None):
    """Convert COCO annotations to YOLO format for person class only"""
    
    print(f"Processing {coco_json_path}...")
    
    # Load COCO annotations
    with open(coco_json_path, 'r') as f:
        coco_data = json.load(f)
    
    # Create mappings
    image_id_to_info = {img['id']: img for img in coco_data['images']}
    category_id_to_name = {cat['id']: cat['name'] for cat in coco_data['categories']}
    
    # Filter only person annotations (category_id = 1 in COCO)
    person_annotations = [ann for ann in coco_data['annotations'] if ann['category_id'] == 1]
    print(f"Found {len(person_annotations)} person annotations")
    
    # Group annotations by image_id
    annotations_by_image = {}
    for ann in person_annotations:
        image_id = ann['image_id']
        if image_id not in annotations_by_image:
            annotations_by_image[image_id] = []
        annotations_by_image[image_id].append(ann)
    
    print(f"Found {len(annotations_by_image)} images with person annotations")
    
    # Process images (limit to max_images if specified)
    image_ids = list(annotations_by_image.keys())
    if max_images:
        image_ids = image_ids[:max_images]
        print(f"Limiting to {max_images} images")
    
    processed_count = 0
    for image_id in image_ids:
        image_info = image_id_to_info.get(image_id)
        if not image_info:
            continue
            
        file_name = image_info['file_name']
        img_width = image_info['width']
        img_height = image_info['height']
        
        # Source and destination paths
        src_image_path = f"{images_source_dir}/{file_name}"
        dst_image_path = f"{images_dest_dir}/{file_name}"
        label_file_path = f"{labels_dest_dir}/{Path(file_name).stem}.txt"
        
        # Skip if source image doesn't exist
        if not os.path.exists(src_image_path):
            continue
        
        # Copy image
        shutil.copy2(src_image_path, dst_image_path)
        
        # Create label file
        with open(label_file_path, 'w') as label_file:
            for ann in annotations_by_image[image_id]:
                if 'segmentation' in ann and ann['segmentation']:
                    # COCO segmentation can be polygon or RLE, we use polygon
                    for segmentation in ann['segmentation']:
                        if isinstance(segmentation, list) and len(segmentation) >= 6:  # Need at least 3 points
                            # Convert polygon to YOLO format (normalized coordinates)
                            normalized_seg = []
                            for i in range(0, len(segmentation), 2):
                                x = segmentation[i] / img_width
                                y = segmentation[i+1] / img_height
                                # Ensure coordinates are within [0,1]
                                x = max(0, min(1, x))
                                y = max(0, min(1, y))
                                normalized_seg.extend([x, y])
                            
                            # Write to label file: class_id x1 y1 x2 y2 ...
                            if len(normalized_seg) >= 6:  # At least 3 points
                                line = "0 " + " ".join([f"{coord:.6f}" for coord in normalized_seg]) + "\n"
                                label_file.write(line)
        
        processed_count += 1
        if processed_count % 100 == 0:
            print(f"Processed {processed_count} images...")
    
    print(f"Completed processing {processed_count} images")
    return processed_count

print("\nConverting validation data for testing...")
# Convert validation data (small subset for quick testing)
val_count = convert_coco_to_yolo_person_only(
    coco_json_path=f"{working_dir}/annotations/instances_val2017.json",
    images_source_dir=f"{coco_root}/val2017",
    images_dest_dir=f"{working_dir}/images/val2017", 
    labels_dest_dir=f"{working_dir}/labels/val2017",
    max_images=200  # Small subset for quick testing
)

print(f"\nConversion completed!")
print(f"Validation images: {val_count}")

# Create dataset YAML file
print("\nCreating dataset YAML file...")
dataset_yaml = {
    'path': working_dir,
    'train': 'images/train2017',  # Not used for validation, but required
    'val': 'images/val2017',
    'nc': 1,
    'names': ['person']
}

yaml_path = f'{working_dir}/coco_person_seg.yaml'
with open(yaml_path, 'w') as f:
    yaml.dump(dataset_yaml, f)

print(f"Dataset YAML created at: {yaml_path}")

# Verify the dataset
def verify_dataset():
    print("\nVerifying dataset...")
    
    # Check validation data  
    val_images = os.listdir(f"{working_dir}/images/val2017")
    val_labels = os.listdir(f"{working_dir}/labels/val2017")
    print(f"Validation images: {len(val_images)}, labels: {len(val_labels)}")
    
    # Check sample label file
    if val_labels:
        sample_label = f"{working_dir}/labels/val2017/{val_labels[0]}"
        print(f"\nSample label file ({val_labels[0]}):")
        with open(sample_label, 'r') as f:
            lines = f.readlines()
            for i, line in enumerate(lines[:2]):  # Show first 2 lines
                print(f"  Line {i+1}: {line.strip()}")
        print(f"  Total segments in this image: {len(lines)}")

verify_dataset()

# Load ORIGINAL YOLOv8-seg model (without fine-tuning)
print("\n" + "="*50)
print("LOADING ORIGINAL YOLOv8-seg MODEL (NO FINE-TUNING)")
print("="*50)

# Load different sizes of original models to compare
original_models = {
    'yolov8n-seg': 'yolov8n-seg.pt',
    'yolov8s-seg': 'yolov8s-seg.pt', 
    'yolov8m-seg': 'yolov8m-seg.pt'
}

# Store results for comparison
results_original = {}

for model_name, model_path in original_models.items():
    print(f"\n\n{'='*60}")
    print(f"EVALUATING: {model_name}")
    print(f"{'='*60}")
    
    try:
        # Load the original pre-trained model
        print(f"Loading {model_path}...")
        model = YOLO(model_path)
        
        # Get model information
        print(f"Model classes: {model.names}")
        print(f"Number of classes: {len(model.names)}")
        
        # Evaluate the model on person class only
        print(f"Evaluating {model_name} on person segmentation...")
        
        # Note: We use the val method but we'll filter results for person class (class 0 in COCO)
        metrics = model.val(
            data=yaml_path,
            split='val',
            save_json=False,
            verbose=True,
            plots=True,
            name=f'original_{model_name}_person_val'
        )
        
        # Store results
        results_original[model_name] = {
            'box_map50': metrics.box.map50,
            'box_map': metrics.box.map,
            'seg_map50': metrics.seg.map50,
            'seg_map': metrics.seg.map,
            'metrics': metrics
        }
        
        print(f"\n=== {model_name} Results on Person Class ===")
        print(f"Detection mAP50: {metrics.box.map50:.4f}")
        print(f"Detection mAP50-95: {metrics.box.map:.4f}")
        print(f"Segmentation mAP50: {metrics.seg.map50:.4f}")
        print(f"Segmentation mAP50-95: {metrics.seg.map:.4f}")
        
        # Test on a few images to visualize
        print(f"\nVisualizing predictions with {model_name}...")
        val_images_dir = f"{working_dir}/images/val2017"
        
        if os.path.exists(val_images_dir):
            val_images = os.listdir(val_images_dir)[:3]  # First 3 images
            
            for img_name in val_images:
                img_path = f"{val_images_dir}/{img_name}"
                
                print(f"Processing: {img_name}")
                
                # Run prediction with original model
                results = model.predict(img_path, conf=0.25, classes=[0])  # Only person class
                
                # Create visualization
                fig, axes = plt.subplots(1, 2, figsize=(12, 6))
                
                # Original image
                orig_img = cv2.imread(img_path)
                orig_img = cv2.cvtColor(orig_img, cv2.COLOR_BGR2RGB)
                axes[0].imshow(orig_img)
                axes[0].set_title(f'Original Image\n{model_name}')
                axes[0].axis('off')
                
                # Prediction
                if len(results[0].boxes) > 0 or len(results[0].masks) > 0:
                    pred_img = results[0].plot()
                    pred_img = cv2.cvtColor(pred_img, cv2.COLOR_BGR2RGB)
                    axes[1].imshow(pred_img)
                    axes[1].set_title(f'{model_name} Prediction\n{len(results[0].boxes)} persons detected')
                else:
                    axes[1].imshow(orig_img)
                    axes[1].set_title(f'{model_name} Prediction\nNo persons detected')
                
                axes[1].axis('off')
                plt.tight_layout()
                plt.show()
                
    except Exception as e:
        print(f"Error evaluating {model_name}: {e}")
        import traceback
        traceback.print_exc()

# Compare results from all original models
print("\n" + "="*70)
print("COMPARISON OF ORIGINAL YOLOv8 MODELS ON PERSON SEGMENTATION")
print("="*70)

print("\nModel Comparison Results:")
print("-" * 80)
print(f"{'Model':<15} {'Det mAP50':<12} {'Det mAP50-95':<15} {'Seg mAP50':<12} {'Seg mAP50-95':<15}")
print("-" * 80)

for model_name, results in results_original.items():
    print(f"{model_name:<15} {results['box_map50']:.4f}      {results['box_map']:.4f}         {results['seg_map50']:.4f}      {results['seg_map']:.4f}")

# Create visualization comparison
if results_original:
    models = list(results_original.keys())
    det_map50 = [results_original[m]['box_map50'] for m in models]
    det_map = [results_original[m]['box_map'] for m in models]
    seg_map50 = [results_original[m]['seg_map50'] for m in models]
    seg_map = [results_original[m]['seg_map'] for m in models]
    
    x = np.arange(len(models))
    width = 0.2
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))
    
    # Detection mAP comparison
    ax1.bar(x - width, det_map50, width, label='mAP50', alpha=0.8)
    ax1.bar(x, det_map, width, label='mAP50-95', alpha=0.8)
    ax1.set_xlabel('Model')
    ax1.set_ylabel('mAP Score')
    ax1.set_title('Detection mAP Comparison (Person Class)')
    ax1.set_xticks(x)
    ax1.set_xticklabels(models)
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # Add value labels on bars
    for i, (v50, v95) in enumerate(zip(det_map50, det_map)):
        ax1.text(i - width, v50 + 0.01, f'{v50:.3f}', ha='center', va='bottom', fontsize=9)
        ax1.text(i, v95 + 0.01, f'{v95:.3f}', ha='center', va='bottom', fontsize=9)
    
    # Segmentation mAP comparison
    ax2.bar(x - width, seg_map50, width, label='mAP50', alpha=0.8)
    ax2.bar(x, seg_map, width, label='mAP50-95', alpha=0.8)
    ax2.set_xlabel('Model')
    ax2.set_ylabel('mAP Score')
    ax2.set_title('Segmentation mAP Comparison (Person Class)')
    ax2.set_xticks(x)
    ax2.set_xticklabels(models)
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    
    # Add value labels on bars
    for i, (v50, v95) in enumerate(zip(seg_map50, seg_map)):
        ax2.text(i - width, v50 + 0.01, f'{v50:.3f}', ha='center', va='bottom', fontsize=9)
        ax2.text(i, v95 + 0.01, f'{v95:.3f}', ha='center', va='bottom', fontsize=9)
    
    plt.tight_layout()
    plt.show()

# Detailed analysis of the best model
if results_original:
    best_model_name = max(results_original.keys(), 
                         key=lambda x: results_original[x]['seg_map'])
    best_results = results_original[best_model_name]
    
    print(f"\n{'='*60}")
    print(f"BEST PERFORMING ORIGINAL MODEL: {best_model_name}")
    print(f"{'='*60}")
    print(f"Detection mAP50: {best_results['box_map50']:.4f}")
    print(f"Detection mAP50-95: {best_results['box_map']:.4f}")
    print(f"Segmentation mAP50: {best_results['seg_map50']:.4f}")
    print(f"Segmentation mAP50-95: {best_results['seg_map']:.4f}")
    
    # Load best model for additional analysis
    best_model = YOLO(original_models[best_model_name])
    
    # Test on more challenging images
    print(f"\nTesting {best_model_name} on diverse person images...")
    val_images_dir = f"{working_dir}/images/val2017"
    
    if os.path.exists(val_images_dir):
        # Get images with different numbers of people
        all_images = os.listdir(val_images_dir)
        
        # Test on 4 different images
        test_images = all_images[:4]
        
        fig, axes = plt.subplots(2, 2, figsize=(15, 12))
        axes = axes.ravel()
        
        for idx, img_name in enumerate(test_images):
            img_path = f"{val_images_dir}/{img_name}"
            
            # Run prediction
            results = best_model.predict(img_path, conf=0.25, classes=[0])
            
            # Original image
            orig_img = cv2.imread(img_path)
            orig_img = cv2.cvtColor(orig_img, cv2.COLOR_BGR2RGB)
            
            # Prediction
            if len(results[0].boxes) > 0 or len(results[0].masks) > 0:
                pred_img = results[0].plot()
                pred_img = cv2.cvtColor(pred_img, cv2.COLOR_BGR2RGB)
                axes[idx].imshow(pred_img)
                axes[idx].set_title(f'{best_model_name}\n{len(results[0].boxes)} persons detected')
            else:
                axes[idx].imshow(orig_img)
                axes[idx].set_title(f'{best_model_name}\nNo persons detected')
            
            axes[idx].axis('off')
        
        plt.tight_layout()
        plt.show()

print("\n" + "="*70)
print("ORIGINAL YOLOv8-seg EVALUATION COMPLETED!")
print("="*70)
print(f"\nSummary:")
print(f"- Evaluated {val_count} validation images")
print(f"- Tested {len(original_models)} different YOLOv8-seg model sizes")
print(f"- All models were evaluated on person segmentation only")
print(f"- Results show baseline performance before fine-tuning")

print("\nThese results serve as the baseline for comparing with your fine-tuned model!")
print("You can now compare these mAP scores with your fine-tuned model's performance.")

In [ ]:
# Install Detectron2 and required dependencies
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install 'git+https://github.com/facebookresearch/detectron2.git'
!pip install opencv-python matplotlib pycocotools albumentations

In [ ]:
# Install Detectron2 and required dependencies
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install 'git+https://github.com/facebookresearch/detectron2.git'
!pip install opencv-python matplotlib pycocotools albumentations

import os
import json
import numpy as np
import cv2
import torch
import torchvision
import matplotlib.pyplot as plt
from PIL import Image
import random
import shutil
from pathlib import Path

# Detectron2 imports
import detectron2
from detectron2.utils.logger import setup_logger
from detectron2 import model_zoo
from detectron2.engine import DefaultPredictor, DefaultTrainer
from detectron2.config import get_cfg
from detectron2.utils.visualizer import Visualizer, ColorMode
from detectron2.data import DatasetCatalog, MetadataCatalog, build_detection_train_loader, build_detection_test_loader
from detectron2.data.datasets import register_coco_instances
from detectron2.evaluation import COCOEvaluator, inference_on_dataset
from detectron2.data import detection_utils as utils

setup_logger()
print(f"Detectron2 version: {detectron2.__version__}")
print(f"PyTorch version: {torch.__version__}")

# Set up paths
coco_root = "/kaggle/input/coco-2017-dataset/coco2017"
working_dir = "/kaggle/working/detectron2-coco-person"
os.makedirs(working_dir, exist_ok=True)

print("Setting up dataset...")

def filter_coco_annotations_for_person(input_json_path, output_json_path, images_dir, max_images=None):
    """Filter COCO annotations to keep only person category and corresponding images"""
    
    print(f"Filtering {input_json_path} for person class only...")
    
    with open(input_json_path, 'r') as f:
        coco_data = json.load(f)
    
    # Person category ID in COCO is 1
    person_category_id = 1
    
    # Filter categories - only keep person
    coco_data['categories'] = [cat for cat in coco_data['categories'] if cat['id'] == person_category_id]
    
    # Filter annotations - only keep person annotations
    person_annotations = [ann for ann in coco_data['annotations'] if ann['category_id'] == person_category_id]
    
    # Get image IDs that have person annotations
    person_image_ids = set(ann['image_id'] for ann in person_annotations)
    
    # Filter images - only keep images with person annotations
    person_images = [img for img in coco_data['images'] if img['id'] in person_image_ids]
    
    # Limit number of images if specified
    if max_images:
        person_images = person_images[:max_images]
        person_image_ids = set(img['id'] for img in person_images)
        person_annotations = [ann for ann in person_annotations if ann['image_id'] in person_image_ids]
    
    # Update COCO data
    coco_data['images'] = person_images
    coco_data['annotations'] = person_annotations
    
    # Save filtered annotations
    with open(output_json_path, 'w') as f:
        json.dump(coco_data, f)
    
    print(f"Filtered dataset: {len(person_images)} images, {len(person_annotations)} annotations")
    return len(person_images), len(person_annotations)

# Create filtered annotations for train and val
train_json_input = f"{coco_root}/annotations/instances_train2017.json"
val_json_input = f"{coco_root}/annotations/instances_val2017.json"

train_json_output = f"{working_dir}/instances_person_train2017.json"
val_json_output = f"{working_dir}/instances_person_val2017.json"

# Filter datasets (using smaller subsets for quick training)
train_img_count, train_ann_count = filter_coco_annotations_for_person(
    train_json_input, train_json_output, 
    f"{coco_root}/train2017",
    max_images=500  # Reduced for quick training
)

val_img_count, val_ann_count = filter_coco_annotations_for_person(
    val_json_input, val_json_output,
    f"{coco_root}/val2017", 
    max_images=100    # Reduced for quick validation
)

print(f"Training set: {train_img_count} images, {train_ann_count} annotations")
print(f"Validation set: {val_img_count} images, {val_ann_count} annotations")

# Register datasets with Detectron2
dataset_name_train = "coco_person_train"
dataset_name_val = "coco_person_val"

register_coco_instances(
    dataset_name_train,
    {},
    train_json_output,
    f"{coco_root}/train2017"
)

register_coco_instances(
    dataset_name_val,
    {}, 
    val_json_output,
    f"{coco_root}/val2017"
)

# Verify dataset registration
print("\nRegistered datasets:")
print(f"Train: {dataset_name_train}")
print(f"Val: {dataset_name_val}")

# Get metadata
metadata_train = MetadataCatalog.get(dataset_name_train)
dataset_dicts_train = DatasetCatalog.get(dataset_name_train)

print(f"\nNumber of training samples: {len(dataset_dicts_train)}")

# Visualize some training samples
def visualize_dataset_samples(dataset_name, num_samples=3):
    """Visualize samples from the dataset"""
    dataset_dicts = DatasetCatalog.get(dataset_name)
    metadata = MetadataCatalog.get(dataset_name)
    
    for i, d in enumerate(random.sample(dataset_dicts, num_samples)):
        img = cv2.imread(d["file_name"])
        visualizer = Visualizer(img[:, :, ::-1], metadata=metadata, scale=0.5)
        vis = visualizer.draw_dataset_dict(d)
        plt.figure(figsize=(12, 8))
        plt.imshow(vis.get_image())
        plt.title(f"Sample {i+1} - {os.path.basename(d['file_name'])}")
        plt.axis('off')
        plt.show()

print("Visualizing training samples...")
visualize_dataset_samples(dataset_name_train, 2)

# Set up configuration for training - FIXED VERSION
print("\nSetting up training configuration...")
cfg = get_cfg()

# Use Mask R-CNN with ResNet-50 backbone
cfg.merge_from_file(model_zoo.get_config_file("COCO-InstanceSegmentation/mask_rcnn_R_50_FPN_3x.yaml"))

# Set dataset
cfg.DATASETS.TRAIN = (dataset_name_train,)
cfg.DATASETS.TEST = (dataset_name_val,)

# Training parameters - adjusted for stability
cfg.DATALOADER.NUM_WORKERS = 2  # Reduced to prevent issues
cfg.MODEL.WEIGHTS = model_zoo.get_checkpoint_url("COCO-InstanceSegmentation/mask_rcnn_R_50_FPN_3x.yaml")
cfg.SOLVER.IMS_PER_BATCH = 2    # Reduced batch size for stability
cfg.SOLVER.BASE_LR = 0.001
cfg.SOLVER.MAX_ITER = 1000      # Reduced for quick training
cfg.SOLVER.STEPS = (500, 750)
cfg.SOLVER.GAMMA = 0.1

# Single class (person) - IMPORTANT: Change class mapping
cfg.MODEL.ROI_HEADS.NUM_CLASSES = 1
cfg.MODEL.RETINANET.NUM_CLASSES = 1

# Evaluation
cfg.TEST.EVAL_PERIOD = 200

# Output directory
cfg.OUTPUT_DIR = f"{working_dir}/output"

# Create output directory
os.makedirs(cfg.OUTPUT_DIR, exist_ok=True)

print("Configuration setup completed!")
print(f"Output directory: {cfg.OUTPUT_DIR}")

# SIMPLIFIED TRAINING - Using DefaultTrainer without custom modifications
print("\nStarting Detectron2 training with DefaultTrainer...")
trainer = DefaultTrainer(cfg)
trainer.resume_or_load(resume=False)

# Train the model
print("Training started...")
try:
    trainer.train()
    print("Training completed successfully!")
except Exception as e:
    print(f"Training error: {e}")
    print("Trying alternative approach...")

# Alternative approach if standard training fails
if not os.path.exists(os.path.join(cfg.OUTPUT_DIR, "model_final.pth")):
    print("\nTrying alternative training configuration...")
    
    # Reset config with simpler settings
    cfg2 = get_cfg()
    cfg2.merge_from_file(model_zoo.get_config_file("COCO-InstanceSegmentation/mask_rcnn_R_50_FPN_1x.yaml"))
    
    # Basic configuration
    cfg2.DATASETS.TRAIN = (dataset_name_train,)
    cfg2.DATASETS.TEST = (dataset_name_val,)
    cfg2.DATALOADER.NUM_WORKERS = 2
    cfg2.MODEL.WEIGHTS = model_zoo.get_checkpoint_url("COCO-InstanceSegmentation/mask_rcnn_R_50_FPN_1x.yaml")
    cfg2.SOLVER.IMS_PER_BATCH = 2
    cfg2.SOLVER.BASE_LR = 0.001
    cfg2.SOLVER.MAX_ITER = 500  # Very short training for testing
    cfg2.MODEL.ROI_HEADS.NUM_CLASSES = 1
    cfg2.OUTPUT_DIR = f"{working_dir}/output_simple"
    os.makedirs(cfg2.OUTPUT_DIR, exist_ok=True)
    
    trainer2 = DefaultTrainer(cfg2)
    trainer2.resume_or_load(resume=False)
    trainer2.train()
    
    # Use the alternative config if successful
    cfg = cfg2

# Load the best model for evaluation
print("\nLoading model for evaluation...")
model_path = os.path.join(cfg.OUTPUT_DIR, "model_final.pth")

if not os.path.exists(model_path):
    # Try alternative path
    model_path = os.path.join(cfg.OUTPUT_DIR, "model_final.pth")
    if not os.path.exists(model_path):
        print("No trained model found. Using pretrained model for demonstration.")
        # Use pretrained model for visualization
        cfg.MODEL.WEIGHTS = model_zoo.get_checkpoint_url("COCO-InstanceSegmentation/mask_rcnn_R_50_FPN_3x.yaml")
else:
    cfg.MODEL.WEIGHTS = model_path

cfg.MODEL.ROI_HEADS.SCORE_THRESH_TEST = 0.5
predictor = DefaultPredictor(cfg)

# Evaluation function
def evaluate_model(predictor, dataset_name):
    """Evaluate the model on the validation set"""
    try:
        print("Evaluating on validation set...")
        evaluator = COCOEvaluator(dataset_name, cfg, False, output_dir=cfg.OUTPUT_DIR)
        val_loader = build_detection_test_loader(cfg, dataset_name)
        results = inference_on_dataset(predictor.model, val_loader, evaluator)
        
        print("\n" + "="*50)
        print("EVALUATION RESULTS")
        print("="*50)
        
        if 'bbox' in results:
            print("Bounding Box Detection Results:")
            print(f"AP: {results['bbox']['AP']:.4f}")
            print(f"AP50: {results['bbox']['AP50']:.4f}")
            print(f"AP75: {results['bbox']['AP75']:.4f}")
        
        if 'segm' in results:
            print("\nSegmentation Results:")
            print(f"AP: {results['segm']['AP']:.4f}")
            print(f"AP50: {results['segm']['AP50']:.4f}")
            print(f"AP75: {results['segm']['AP75']:.4f}")
            
        return results
    except Exception as e:
        print(f"Evaluation failed: {e}")
        return None

# Run evaluation
results = evaluate_model(predictor, dataset_name_val)

# Visualization function
def visualize_predictions(predictor, dataset_name, num_samples=4):
    """Visualize predictions on validation samples"""
    dataset_dicts = DatasetCatalog.get(dataset_name)
    metadata = MetadataCatalog.get(dataset_name)
    
    samples = random.sample(dataset_dicts, min(num_samples, len(dataset_dicts)))
    
    for i, d in enumerate(samples):
        # Load image
        im = cv2.imread(d["file_name"])
        
        # Run prediction
        outputs = predictor(im)
        
        # Create visualization
        fig, axes = plt.subplots(1, 3, figsize=(18, 6))
        
        # Original image
        axes[0].imshow(im[:, :, ::-1])
        axes[0].set_title("Original Image")
        axes[0].axis('off')
        
        # Ground truth
        visualizer = Visualizer(im[:, :, ::-1], metadata=metadata, scale=0.8)
        vis_gt = visualizer.draw_dataset_dict(d)
        axes[1].imshow(vis_gt.get_image())
        axes[1].set_title("Ground Truth")
        axes[1].axis('off')
        
        # Predictions
        visualizer = Visualizer(im[:, :, ::-1], metadata=metadata, scale=0.8)
        vis_pred = visualizer.draw_instance_predictions(outputs["instances"].to("cpu"))
        axes[2].imshow(vis_pred.get_image())
        axes[2].set_title("Predictions")
        axes[2].axis('off')
        
        plt.tight_layout()
        plt.show()
        
        # Print detection info
        instances = outputs["instances"]
        if len(instances) > 0:
            print(f"Sample {i+1}: Detected {len(instances)} persons")
        else:
            print(f"Sample {i+1}: No persons detected")
        print("-" * 50)

print("\nVisualizing predictions on validation set...")
visualize_predictions(predictor, dataset_name_val, 4)

# Performance analysis
def analyze_performance(predictor, dataset_name, num_images=5):
    """Analyze model performance on sample images"""
    dataset_dicts = DatasetCatalog.get(dataset_name)
    metadata = MetadataCatalog.get(dataset_name)
    
    samples = random.sample(dataset_dicts, min(num_images, len(dataset_dicts)))
    
    print(f"\nPerformance Analysis on {len(samples)} images:")
    print("=" * 60)
    
    total_gt = 0
    total_pred = 0
    correct_detections = 0
    
    for i, d in enumerate(samples):
        im = cv2.imread(d["file_name"])
        
        # Ground truth
        gt_annotations = d["annotations"]
        gt_count = len(gt_annotations)
        total_gt += gt_count
        
        # Prediction
        outputs = predictor(im)
        instances = outputs["instances"]
        pred_count = len(instances)
        total_pred += pred_count
        
        # Simple correctness check (for demo purposes)
        # In real evaluation, we'd use IoU metrics
        if gt_count > 0 and pred_count > 0:
            correct_detections += min(gt_count, pred_count)
        
        print(f"Image {i+1}: GT={gt_count}, Pred={pred_count}")
    
    if total_gt > 0:
        detection_rate = correct_detections / total_gt
        print(f"\nOverall Detection Rate: {detection_rate:.2f} ({correct_detections}/{total_gt})")
    else:
        print("\nNo ground truth annotations found in samples")

analyze_performance(predictor, dataset_name_val, 5)

# Test with different confidence thresholds
def test_confidence_thresholds(predictor, dataset_name):
    """Test model with different confidence thresholds"""
    print("\nTesting different confidence thresholds:")
    print("=" * 50)
    
    dataset_dicts = DatasetCatalog.get(dataset_name)
    sample = random.choice(dataset_dicts)
    im = cv2.imread(sample["file_name"])
    
    thresholds = [0.3, 0.5, 0.7]
    
    fig, axes = plt.subplots(1, len(thresholds) + 1, figsize=(20, 5))
    
    # Original image
    axes[0].imshow(im[:, :, ::-1])
    axes[0].set_title("Original")
    axes[0].axis('off')
    
    for i, threshold in enumerate(thresholds):
        # Update threshold
        cfg.MODEL.ROI_HEADS.SCORE_THRESH_TEST = threshold
        predictor = DefaultPredictor(cfg)
        
        # Predict
        outputs = predictor(im)
        
        # Visualize
        visualizer = Visualizer(im[:, :, ::-1], metadata=MetadataCatalog.get(dataset_name), scale=0.8)
        vis = visualizer.draw_instance_predictions(outputs["instances"].to("cpu"))
        
        axes[i+1].imshow(vis.get_image())
        axes[i+1].set_title(f"Thresh={threshold}\nDetected: {len(outputs['instances'])}")
        axes[i+1].axis('off')
        
        print(f"Threshold {threshold}: {len(outputs['instances'])} detections")
    
    plt.tight_layout()
    plt.show()

test_confidence_thresholds(predictor, dataset_name_val)

# Save training summary
training_summary = {
    "dataset": {
        "train_images": train_img_count,
        "train_annotations": train_ann_count,
        "val_images": val_img_count,
        "val_annotations": val_ann_count
    },
    "model": {
        "architecture": "Mask R-CNN R50-FPN",
        "classes": 1,
        "output_dir": cfg.OUTPUT_DIR
    },
    "training": {
        "iterations": cfg.SOLVER.MAX_ITER,
        "batch_size": cfg.SOLVER.IMS_PER_BATCH
    }
}

with open(f"{working_dir}/training_summary.json", "w") as f:
    json.dump(training_summary, f, indent=2)

print("\n" + "="*70)
print("DETECTRON2 TRAINING SUMMARY")
print("="*70)
print(f"Training images: {train_img_count}")
print(f"Validation images: {val_img_count}")
print(f"Model: Mask R-CNN with ResNet-50-FPN")
print(f"Output directory: {cfg.OUTPUT_DIR}")

if results:
    print(f"Segmentation mAP50: {results['segm']['AP50']:.4f}")
    print(f"Segmentation mAP: {results['segm']['AP']:.4f}")

print("\nNext steps:")
print("1. Increase max_images parameter for full training")
print("2. Increase SOLVER.MAX_ITER for better performance")
print("3. Use larger batch size if GPU memory allows")
print("4. Try different model architectures")

print("\n🎉 Detectron2 training pipeline completed!")

In [ ]:
# import os
# import sys
# import time
# import json
# from pathlib import Path
# from typing import List, Tuple
# import numpy as np
# from PIL import Image
# import cv2
# import torch
# from transformers import AutoProcessor, PaliGemmaProcessor, PaliGemmaForConditionalGeneration
# from ultralytics import SAM
# from pycocotools.coco import COCO
# from pycocotools.cocoeval import COCOeval
# from pycocotools import mask as maskUtils
# from datasets import Dataset
# import albumentations as A
# from peft import LoraConfig, get_peft_model
# from trl import SFTTrainer, SFTConfig
# from huggingface_hub import login
# import re
# from functools import partial
# import supervision as sv
# from torchvision.ops import nms
# from torch import Tensor
# from ultralytics import YOLO
# from pycocotools.coco import COCO
# from pycocotools.cocoeval import COCOeval
# from pycocotools import mask as maskUtils

# # ObjectClear
# sys.path.append('/kaggle/working/ObjectClear')
# from objectclear.pipelines import ObjectClearPipeline
# from objectclear.utils import resize_by_short_side

# DEV = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# # Disable W&B for YOLO
# os.environ["WANDB_MODE"] = "disabled"

# # Helper functions
# def pil_to_cv(img): return cv2.cvtColor(np.array(img), cv2.COLOR_RGB2BGR)
# def cv_to_pil(img): return Image.fromarray(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))

# def overlay_mask_on_image(img: np.ndarray, masks: List[np.ndarray], alpha=0.5):
#     out = img.copy()
#     if len(masks)==0: return out
#     rng = np.random.RandomState(0)
#     for m in masks:
#         col = rng.randint(0,255,3).tolist()
#         mask_bool = m.astype(bool)
#         out[mask_bool] = (out[mask_bool].astype(np.float32)*(1-alpha) + np.array(col)*(alpha)).astype(np.uint8)
#     return out

# def make_grid(images: List[np.ndarray], cols=3, pad=8):
#     rows = 1
#     h = max(im.shape[0] for im in images)
#     w = max(im.shape[1] for im in images)
#     H = rows*h + pad*(rows+1)
#     W = cols*w + pad*(cols+1)
#     canvas = np.ones((H,W,3), dtype=np.uint8)*255
#     idx=0
#     for c in range(cols):
#         if idx>=len(images): break
#         im = images[idx]
#         ih,iw = im.shape[:2]
#         ys = pad + (h-ih)//2
#         xs = pad + c*(w+pad) + (w-iw)//2
#         canvas[ys:ys+ih, xs:xs+iw] = im
#         idx+=1
#     return canvas

# def masks_to_coco_rle(mask):
#     rle = maskUtils.encode(np.asfortranarray(mask.astype(np.uint8)))
#     if isinstance(rle.get("counts"), bytes):
#         rle["counts"] = rle["counts"].decode("utf-8")
#     return rle

# # Prepare filtered COCO dataset for person only
# def prepare_dataset(num_train_images=300):
#     # Train
#     coco_train = COCO("/kaggle/input/coco-2017-dataset/coco2017/annotations/instances_train2017.json")
#     cat_ids = coco_train.getCatIds(catNms=['person'])
#     img_ids = coco_train.getImgIds(catIds=cat_ids)
#     sampled_img_ids = np.random.choice(img_ids, num_train_images, replace=False).tolist()
#     images = coco_train.loadImgs(sampled_img_ids)
#     anns = coco_train.loadAnns(coco_train.getAnnIds(imgIds=sampled_img_ids, catIds=cat_ids))
#     categories = coco_train.loadCats(cat_ids)
#     categories[0]['id'] = 0
#     for ann in anns:
#         ann['category_id'] = 0
#     filtered_dict = {'images': images, 'annotations': anns, 'categories': categories}
#     os.makedirs('/kaggle/working/datasets/coco/annotations', exist_ok=True)
#     with open('/kaggle/working/datasets/coco/annotations/instances_train2017.json', 'w') as f:
#         json.dump(filtered_dict, f)

#     # Val (full for person)
#     coco_val = COCO("/kaggle/input/coco-2017-dataset/coco2017/annotations/instances_val2017.json")
#     val_img_ids = coco_val.getImgIds(catIds=cat_ids)
#     val_images = coco_val.loadImgs(val_img_ids)
#     val_anns = coco_val.loadAnns(coco_val.getAnnIds(imgIds=val_img_ids, catIds=cat_ids))
#     val_categories = coco_val.loadCats(cat_ids)
#     val_categories[0]['id'] = 0
#     for ann in val_anns:
#         ann['category_id'] = 0
#     val_dict = {'images': val_images, 'annotations': val_anns, 'categories': val_categories}
#     with open('/kaggle/working/datasets/coco/annotations/instances_val2017.json', 'w') as f:
#         json.dump(val_dict, f)

#     # Symlinks for images
#     if not os.path.exists('/kaggle/working/datasets/coco/train2017'):
#         os.symlink('/kaggle/input/coco-2017-dataset/coco2017/train2017', '/kaggle/working/datasets/coco/train2017')
#     if not os.path.exists('/kaggle/working/datasets/coco/val2017'):
#         os.symlink('/kaggle/input/coco-2017-dataset/coco2017/val2017', '/kaggle/working/datasets/coco/val2017')

#     # Create YAML
#     yaml_content = """
# path: /kaggle/working/datasets/coco
# train: train2017
# val: val2017
# nc: 1
# names:
#   0: person
# """
#     with open('/kaggle/working/coco_person.yaml', 'w') as f:
#         f.write(yaml_content)

# # Fine-tune YOLOv8-seg
# def fine_tune_yolo(epochs=3):
#     model = YOLO('/kaggle/input/yolo11-seg/pytorch/default/1/yolo11n-seg.pt')
#     model.train(data='/kaggle/working/coco_person.yaml', epochs=epochs, imgsz=640, batch=8, device=0)

# # Load ObjectClear
# def load_objectclear():
#     pipe = ObjectClearPipeline.from_pretrained_with_custom_modules(
#         "jixin0101/ObjectClear",
#         torch_dtype=torch.float16 if 'cuda' in str(DEV) else torch.float32,
#         apply_attention_guided_fusion=True,
#         variant="fp16" if 'cuda' in str(DEV) else None
#     ).to(DEV)
#     return pipe

# # Process and evaluate
# def process_and_evaluate(coco_root: str, ann_file: str, out_dir: str,
#                          num_images: int = 100, box_th=0.15):
#     os.makedirs(out_dir, exist_ok=True)
#     coco = COCO(ann_file)  # Original ann for eval
#     img_ids = coco.getImgIds(catIds=[1])[:num_images]
#     print(f"Processing {len(img_ids)} images (images containing person).")

#     prepare_dataset(num_train_images=300)
#     fine_tune_yolo(epochs=3)

#     model = YOLO('/kaggle/working/runs/segment/train/weights/best.pt')  # Adjust if multiple runs
#     objclear = load_objectclear()
#     coco_dt = []

#     for img_id in img_ids:
#         info = coco.loadImgs([img_id])[0]
#         img_path = os.path.join(coco_root, info['file_name'])
#         img_pil = Image.open(img_path).convert("RGB")
#         W, H = img_pil.size
#         image_np = np.array(img_pil)

#         results = model(image_np, conf=box_th, iou=0.5, verbose=False)
#         if not results or results[0].masks is None:
#             print(f"img {img_id}: no boxes")
#             continue

#         masks_tensor = results[0].masks.data.cpu().numpy()  # (N, H, W)
#         scores = results[0].boxes.conf.cpu().numpy().tolist()

#         if len(masks_tensor) == 0:
#             print(f"img {img_id}: no masks")
#             continue

#         combined_mask = np.any(masks_tensor > 0.5, axis=0).astype(np.uint8)

#         mask_pil = Image.fromarray((combined_mask * 255).astype("uint8"))
#         try:
#             cleared_pil = run_objectclear(objclear, img_pil, mask_pil)
#         except Exception as e:
#             print(f"img {img_id}: ObjectClear failed: {e}")
#             continue

#         gt_ann_ids = coco.getAnnIds(imgIds=[img_id], catIds=[1], iscrowd=None)
#         gt_mask = coco_mask_to_binary(coco, gt_ann_ids, H, W)

#         intersection = np.logical_and(gt_mask, combined_mask).sum()
#         union = np.logical_or(gt_mask, combined_mask).sum()
#         iou_score = intersection / (union + 1e-6)
#         print(f"img {img_id}: IoU(YOLO vs GT) = {iou_score:.3f}")

#         orig_cv = pil_to_cv(img_pil)
#         gt_cv = overlay_mask_on_image(orig_cv, [gt_mask], alpha=0.5)
#         yolo_masks = [(m > 0.5).astype(np.uint8) for m in masks_tensor]
#         yolo_cv = overlay_mask_on_image(orig_cv, yolo_masks, alpha=0.5)
#         cleared_cv = pil_to_cv(cleared_pil)
#         grid = make_grid([gt_cv, yolo_cv, cleared_cv], cols=3)
#         cv2.imwrite(os.path.join(out_dir, f"{img_id:012d}_segm.jpg"), grid)

#         for i, m in enumerate(masks_tensor):
#             mask_bin = (m > 0.5).astype(np.uint8)
#             rle = masks_to_coco_rle(mask_bin)
#             coco_dt.append({
#                 "image_id": int(img_id),
#                 "category_id": 1,
#                 "segmentation": rle,
#                 "score": scores[i]
#             })

#     if len(coco_dt) == 0:
#         print("⚠️ No predictions generated.")
#         return 0.0

#     dt_path = os.path.join(out_dir, "preds_coco_dt.json")
#     with open(dt_path, "w") as f:
#         json.dump(coco_dt, f)

#     coco_dt_obj = coco.loadRes(dt_path)
#     coco_eval = COCOeval(coco, coco_dt_obj, iouType="segm")
#     coco_eval.params.iouThrs = np.array([0.5])
#     coco_eval.params.imgIds = img_ids
#     coco_eval.evaluate()
#     coco_eval.accumulate()
#     coco_eval.summarize()

#     map_mask = float(coco_eval.stats[0])
#     print(f"\n📊 Segmentation Results:")
#     print(f" - Mask mAP@[.5:.95] = {map_mask:.4f}")  # Note: With iouThrs=[0.5], this is AP@0.5
#     return map_mask


# # ObjectClear
# def run_objectclear(pipe, image_pil: Image.Image, mask_pil: Image.Image):
#     img_proc = resize_by_short_side(image_pil, 512, Image.BICUBIC)
#     mask_proc = resize_by_short_side(mask_pil, 512, Image.NEAREST)
#     use_cuda = 'cuda' in str(DEV)
#     with torch.autocast(device_type='cuda', enabled=use_cuda):
#         out = pipe(
#             prompt="remove the instance of object",
#             image=img_proc,
#             mask_image=mask_proc,
#             generator=torch.Generator(DEV).manual_seed(42),
#             num_inference_steps=20,
#             strength=0.99,
#             guidance_scale=2.5,
#             height=img_proc.size[1],
#             width=img_proc.size[0],
#         )
#     return out.images[0].resize(image_pil.size)

# # Coco mask to binary
# def coco_mask_to_binary(coco, ann_ids, H, W):
#     anns = coco.loadAnns(ann_ids)
#     mask = np.zeros((H, W), dtype=np.uint8)
#     for ann in anns:
#         m = coco.annToMask(ann)
#         mask = np.clip(mask + m, 0, 1)
#     return mask

# # Main execution
# coco_root = "/kaggle/input/coco-2017-dataset/coco2017/val2017"
# ann_file = "/kaggle/input/coco-2017-dataset/coco2017/annotations/instances_val2017.json"
# out_dir = "/kaggle/working/results"
# os.makedirs(out_dir, exist_ok=True)
# t0 = time.time()
# map_mask = process_and_evaluate(coco_root, ann_file, out_dir, 100, 0.15)
# print("Total time:", time.time() - t0)

# # Note: This code fine-tunes YOLOv8-seg on 600 training images (30% of 2000) for the 'person' category with 1 epoch. It uses filtered COCO annotations for person only. Run your original code to get the baseline mAP, then this to compare and verify >2% improvement. If runs/segment/train exists from previous, you may need to adjust the path or remove old runs.

In [ ]:
# Install necessary dependencies if needed
!pip install -Uq ultralytics pycocotools

# Import libraries
import os
import sys
import time
import json
import numpy as np
from typing import List
from PIL import Image
import cv2
import torch
from pycocotools.coco import COCO
import shutil
import random
from ultralytics import YOLO
from pycocotools.cocoeval import COCOeval
from pycocotools import mask as maskUtils
from IPython.display import display

if not os.path.exists('/kaggle/working/ObjectClear'):
    !git clone https://github.com/jixin0101/ObjectClear.git /kaggle/working/ObjectClear

sys.path.append('/kaggle/working/ObjectClear')
from objectclear.pipelines import ObjectClearPipeline
from objectclear.utils import resize_by_short_side

DEV = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Disable W&B for YOLO
os.environ["WANDB_MODE"] = "disabled"

# Helper functions
def pil_to_cv(img): return cv2.cvtColor(np.array(img), cv2.COLOR_RGB2BGR)
def cv_to_pil(img): return Image.fromarray(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))

def overlay_mask_on_image(img: np.ndarray, masks: List[np.ndarray], alpha=0.5):
    out = img.copy()
    if len(masks) == 0: return out
    rng = np.random.RandomState(0)
    for m in masks:
        col = rng.randint(0, 255, 3).tolist()
        mask_bool = m.astype(bool)
        out[mask_bool] = (out[mask_bool].astype(np.float32) * (1 - alpha) + np.array(col) * (alpha)).astype(np.uint8)
    return out

def make_grid(images: List[np.ndarray], cols=3, pad=8):
    rows = (len(images) + cols - 1) // cols
    h = max(im.shape[0] for im in images) if images else 0
    w = max(im.shape[1] for im in images) if images else 0
    H = rows * h + pad * (rows + 1)
    W = cols * w + pad * (cols + 1)
    canvas = np.ones((H, W, 3), dtype=np.uint8) * 255
    idx = 0
    for r in range(rows):
        for c in range(cols):
            if idx >= len(images): return canvas
            im = images[idx]
            ih, iw = im.shape[:2]
            ys = pad + r * (h + pad) + (h - ih) // 2
            xs = pad + c * (w + pad) + (w - iw) // 2
            canvas[ys:ys + ih, xs:xs + iw] = im
            idx += 1
    return canvas

def masks_to_coco_rle(mask):
    rle = maskUtils.encode(np.asfortranarray(mask.astype(np.uint8)))
    if isinstance(rle.get("counts"), bytes):
        rle["counts"] = rle["counts"].decode("utf-8")
    return rle

# Fine-tune YOLOv8-seg with callback to track metrics
class MetricsCallback:
    def __init__(self):
        self.train_losses = []
        self.val_maps = []

    def on_fit_epoch_end(self, trainer):
        if hasattr(trainer, "tloss"):
            self.train_losses.append(trainer.tloss.sum().item())
        else:
            self.train_losses.append(0.0)
    
        if hasattr(trainer.validator, "metrics"):
            metrics = trainer.validator.metrics
            self.val_maps.append(metrics.seg.map50.item())
        else:
            self.val_maps.append(0.0)

# Fine-tune YOLOv8-seg
def fine_tune_yolo(epochs=30, callback=None):
    model = YOLO('yolov8n-seg.pt')
    if callback:
        model.add_callback('on_fit_epoch_end', callback.on_fit_epoch_end)
    results = model.train(data='/kaggle/working/coco_person.yaml', epochs=epochs, imgsz=1024, batch=8, device=0, plots=False, name='train_person')
    return model

# Load ObjectClear
def load_objectclear():
    pipe = ObjectClearPipeline.from_pretrained_with_custom_modules(
        "jixin0101/ObjectClear",
        torch_dtype=torch.float16 if 'cuda' in str(DEV) else torch.float32,
        apply_attention_guided_fusion=True,
        variant="fp16" if 'cuda' in str(DEV) else None
    ).to(DEV)
    return pipe

def prepare_dataset(
    coco_root='/kaggle/input/coco-2017-dataset/coco2017',
    category_name='person',
    num_train_images=50,
    num_val_images=10,
    output_dir='/kaggle/working/datasets/coco'
):
    if os.path.exists(output_dir):
        shutil.rmtree(output_dir)
    os.makedirs(output_dir, exist_ok=True)
    train_img_dir = os.path.join(output_dir, 'images/train2017')
    val_img_dir = os.path.join(output_dir, 'images/val2017')
    train_lbl_dir = os.path.join(output_dir, 'labels/train2017')
    val_lbl_dir = os.path.join(output_dir, 'labels/val2017')

    for d in [train_img_dir, val_img_dir, train_lbl_dir, val_lbl_dir]:
        os.makedirs(d, exist_ok=True)

    def get_valid_image_ids(coco, cat_id, num_images):
        img_ids = coco.getImgIds(catIds=[cat_id])
        random.shuffle(img_ids)
        valid_img_ids = []
        for img_id in img_ids:
            anns = coco.loadAnns(coco.getAnnIds(imgIds=[img_id], catIds=[cat_id], iscrowd=False))
            valid_anns = [ann for ann in anns if 'segmentation' in ann and isinstance(ann['segmentation'], list) and len(ann['segmentation']) > 0 and ann['bbox'][2] > 1 and ann['bbox'][3] > 1]
            if valid_anns:
                valid_img_ids.append(img_id)
            if len(valid_img_ids) >= num_images:
                break
        return valid_img_ids

    def save_labels(coco, img_ids, split, cat_id):
        img_dir = train_img_dir if split == 'train' else val_img_dir
        lbl_dir = train_lbl_dir if split == 'train' else val_lbl_dir
        count = 0
        for img_id in img_ids:
            img_info = coco.loadImgs([img_id])[0]
            img_filename = img_info['file_name']
            src_img_path = os.path.join(coco_root, f'{split}2017', img_filename)
            dst_img_path = os.path.join(img_dir, img_filename)
            if not os.path.exists(src_img_path):
                continue
            anns = coco.loadAnns(coco.getAnnIds(imgIds=[img_id], catIds=[cat_id], iscrowd=False))
            lines = []
            for ann in anns:
                segs = ann['segmentation']
                if not isinstance(segs, list) or len(segs) == 0:
                    continue
                if ann['bbox'][2] <= 1 or ann['bbox'][3] <= 1:
                    continue
                # Take the first (usually main) polygon
                seg = segs[0]
                if len(seg) < 6:
                    continue
                norm_seg = [
                    seg[i] / img_info['width'] if i % 2 == 0 else seg[i] / img_info['height']
                    for i in range(len(seg))
                ]
                lines.append("0 " + " ".join([f"{x:.6f}" for x in norm_seg]))
            if lines:
                shutil.copy(src_img_path, dst_img_path)
                lbl_path = os.path.join(lbl_dir, img_filename.replace('.jpg', '.txt'))
                with open(lbl_path, 'w') as f:
                    f.write('\n'.join(lines))
                count += 1
        return count

    coco_train = COCO(os.path.join(coco_root, 'annotations/instances_train2017.json'))
    coco_val = COCO(os.path.join(coco_root, 'annotations/instances_val2017.json'))
    cat_id = coco_train.getCatIds(catNms=[category_name])[0]

    train_ids = get_valid_image_ids(coco_train, cat_id, num_train_images)
    val_ids = get_valid_image_ids(coco_val, cat_id, num_val_images)

    n_train = save_labels(coco_train, train_ids, 'train', cat_id)
    n_val = save_labels(coco_val, val_ids, 'val', cat_id)

    print("✅ Dataset created.")
    print(f"- Train images with valid masks: {n_train}")
    print(f"- Val images with valid masks: {n_val}")

# Create YAML file
def create_yaml():
    yaml_content = """
train: /kaggle/working/datasets/coco/images/train2017
val: /kaggle/working/datasets/coco/images/val2017

nc: 1
names: ['person']
"""
    with open('/kaggle/working/coco_person.yaml', 'w') as f:
        f.write(yaml_content)

# ObjectClear
def run_objectclear(pipe, image_pil: Image.Image, mask_pil: Image.Image):
    img_proc = resize_by_short_side(image_pil, 512, Image.BICUBIC)
    mask_proc = resize_by_short_side(mask_pil, 512, Image.NEAREST)
    use_cuda = 'cuda' in str(DEV)
    with torch.autocast(device_type='cuda' if use_cuda else 'cpu', enabled=use_cuda):
        out = pipe(
            prompt="remove the instance of object",
            image=img_proc,
            mask_image=mask_proc,
            generator=torch.Generator(DEV).manual_seed(42),
            num_inference_steps=20,
            strength=0.99,
            guidance_scale=2.5,
            height=img_proc.size[1],
            width=img_proc.size[0],
        )
    return out.images[0].resize(image_pil.size, Image.LANCZOS)

# Coco mask to binary
def coco_mask_to_binary(coco, ann_ids, H, W):
    anns = coco.loadAnns(ann_ids)
    mask = np.zeros((H, W), dtype=np.uint8)
    for ann in anns:
        m = coco.annToMask(ann)
        mask = np.clip(mask + m, 0, 1)
    return mask

def process_and_evaluate(coco_img_dir, ann_file, out_dir, num_images, box_th=0.05):
    coco = COCO(ann_file)
    cat_id = coco.getCatIds(catNms=['person'])[0]
    img_ids = coco.getImgIds(catIds=[cat_id])
    random.shuffle(img_ids)
    img_ids = img_ids[:num_images]
    print(f"Processing {num_images} images (images containing person).")

    model = YOLO('/kaggle/working/runs/segment/train_person/weights/best.pt')

    pred_anns = []
    visuals = []
    no_pred_count = 0
    for img_id in img_ids:
        img_info = coco.loadImgs(img_id)[0]
        img_path = os.path.join(coco_img_dir, img_info['file_name'])
        orig_image_pil = Image.open(img_path).convert("RGB")
        results = model(img_path, conf=box_th, imgsz=640)[0]  # Ensure consistent size
        if results.masks is None or len(results.masks) == 0:
            print(f"img {img_id}: no boxes")
            no_pred_count += 1
            continue
        masks = results.masks.data.cpu().numpy()
        boxes = results.boxes.xyxy.cpu().numpy()
        confs = results.boxes.conf.cpu().numpy()
        
        # Resize masks to original image size
        resized_masks = []
        for mask in masks:
            mask_pil = Image.fromarray((mask > 0.5).astype(np.uint8) * 255)
            resized_mask_pil = mask_pil.resize((img_info['width'], img_info['height']), Image.NEAREST)
            resized_masks.append(np.array(resized_mask_pil) / 255)
        
        for i in range(len(resized_masks)):
            mask_bin = resized_masks[i]
            rle = masks_to_coco_rle(mask_bin)
            bbox = [float(x) for x in boxes[i]]
            score = float(confs[i])
            area = int(np.sum(mask_bin))
            pred_ann = {
                "id": len(pred_anns) + 1,
                "image_id": img_id,
                "category_id": 1,
                "segmentation": rle,
                "area": area,
                "bbox": bbox,
                "iscrowd": 0,
                "score": score
            }
            pred_anns.append(pred_ann)
        
        # For removal: union of resized masks
        if len(resized_masks) > 0:
            union_mask_np = np.any(np.stack(resized_masks) > 0.5, axis=0).astype(np.uint8) * 255
            mask_pil = Image.fromarray(union_mask_np)
            cleared = run_objectclear(pipe, orig_image_pil, mask_pil)
            cleared.save(os.path.join(out_dir, img_info['file_name'].replace('.jpg', '_cleared.jpg')))
            
            # Visualization
            img_cv = pil_to_cv(orig_image_pil)
            gt_ann_ids = coco.getAnnIds(imgIds=img_id, catIds=cat_id, iscrowd=False)
            gt_mask = coco_mask_to_binary(coco, gt_ann_ids, img_info['height'], img_info['width'])
            gt_overlay = overlay_mask_on_image(img_cv, [gt_mask])
            pred_overlay = overlay_mask_on_image(img_cv, [union_mask_np // 255])
            cleared_cv = pil_to_cv(cleared)
            grid = make_grid([img_cv, gt_overlay, pred_overlay, cleared_cv], cols=4)
            visuals.append(cv_to_pil(grid))

    if no_pred_count == num_images:
        print("⚠️ No predictions generated.")
        return None

    # Evaluate mAP for masks
    dt_json = "/kaggle/working/dt.json"
    with open(dt_json, 'w') as f:
        json.dump(pred_anns, f)
    gt = COCO(ann_file)
    dt = gt.loadRes(dt_json)
    cocoEval = COCOeval(gt, dt, "segm")
    cocoEval.params.imgIds = img_ids
    cocoEval.params.catIds = [cat_id]
    cocoEval.evaluate()
    cocoEval.accumulate()
    cocoEval.summarize()
    map_mask = cocoEval.stats[0]  # AP across IoU
    
    
    return map_mask

# Main execution
prepare_dataset()
create_yaml()
callback = MetricsCallback()
model = fine_tune_yolo(epochs=30, callback=callback)
train_losses = callback.train_losses
val_maps = callback.val_maps
pipe = load_objectclear()
coco_img_dir = "/kaggle/input/coco-2017-dataset/coco2017/val2017"
ann_file = "/kaggle/input/coco-2017-dataset/coco2017/annotations/instances_val2017.json"
out_dir = "/kaggle/working/results"
os.makedirs(out_dir, exist_ok=True)
t0 = time.time()
map_mask = process_and_evaluate(coco_img_dir, ann_file, out_dir, num_images=100, box_th=0.05)
print("Total time:", time.time() - t0)


In [ ]:
# import json
# import os
# from PIL import Image, ImageDraw

# # Paths
# json_path = "/kaggle/input/coco-2017-dataset/coco2017/annotations/instances_val2017.json"
# image_dir = "/kaggle/input/coco-2017-dataset/coco2017/val2017/"

# target_image_id = 404839

# # Load COCO JSON
# with open(json_path, 'r') as f:
#     coco_data = json.load(f)

# # Get human annotations for the image
# human_annotations = [
#     ann for ann in coco_data['annotations']
#     if ann['image_id'] == target_image_id and ann['category_id'] == 1
# ]

# if not human_annotations:
#     print("No human annotations found for image.")
# else:
#     # Load image info and open image
#     image_info = next(img for img in coco_data['images'] if img['id'] == target_image_id)
#     file_name = image_info['file_name']
#     image_path = os.path.join(image_dir, file_name)
#     img = Image.open(image_path).convert("RGBA")

#     # Create an overlay image for segmentation masks with transparency
#     overlay = Image.new("RGBA", img.size, (255,255,255,0))
#     draw = ImageDraw.Draw(overlay)

#     # Draw segmentation polygons
#     for ann in human_annotations:
#         segmentation = ann['segmentation']
#         # segmentation is a list of polygons (each polygon is a list of points)
#         for polygon in segmentation:
#             # polygon is a flat list: [x1,y1,x2,y2,...]
#             xy = [(polygon[i], polygon[i+1]) for i in range(0, len(polygon), 2)]
#             draw.polygon(xy, outline=(255,0,0,255), fill=(255,0,0,100))  # red with alpha 100

#     # Composite original image with overlay
#     img_with_mask = Image.alpha_composite(img, overlay)

#     # Convert back to RGB and show
#     img_with_mask = img_with_mask.convert("RGB")
# img_with_mask


In [ ]:
# from ultralytics import YOLO
# from PIL import Image
# import numpy as np
# import matplotlib.pyplot as plt

# model = YOLO('yolov8n-seg.pt')

# image_path = "/kaggle/input/coco-2017-dataset/coco2017/val2017/000000404839.jpg"
# results = model(image_path)[0]

# input_height, input_width = results.masks.data.shape[1], results.masks.data.shape[2]

# img = Image.open(image_path).convert("RGB")
# img_resized = img.resize((input_width, input_height))
# img_np = np.array(img_resized)

# masks = results.masks.data.cpu().numpy()
# class_ids = results.boxes.cls.cpu().numpy()

# # Filter only person class (class_id == 0)
# person_indices = [i for i, cls in enumerate(class_ids) if cls == 0]

# colors = np.random.randint(0, 255, size=(len(person_indices), 3))
# overlay = np.zeros_like(img_np, dtype=np.uint8)

# for i, idx in enumerate(person_indices):
#     mask = masks[idx]
#     color = colors[i]
#     overlay[mask > 0] = color

# alpha = 0.5
# blended = img_np * (1 - alpha) + overlay * alpha
# blended = blended.astype(np.uint8)

# plt.figure(figsize=(12,12))
# plt.imshow(blended)
# plt.axis('off')
# plt.title("YOLOv8-seg Predicted Human Masks Only")
# plt.show()


In [ ]:
# from pycocotools.coco import COCO
# from pycocotools import mask as maskUtils
# from PIL import Image, ImageDraw
# import numpy as np
# import matplotlib.pyplot as plt

# # Path to COCO annotations JSON file
# annFile = "/kaggle/input/coco-2017-dataset/coco2017/annotations/instances_val2017.json"

# # Load COCO annotations
# coco = COCO(annFile)

# # Path to image
# image_path = "/kaggle/input/coco-2017-dataset/coco2017/val2017/000000117719.jpg"
# img = Image.open(image_path)

# # Get image ID from COCO (match by file name)
# file_name = image_path.split('/')[-1]

# img_ids = coco.getImgIds()
# imgs = coco.loadImgs(img_ids)

# img_info = next(img for img in imgs if img['file_name'] == file_name)
# image_id = img_info['id']


# # Get annotation IDs for that image
# ann_ids = coco.getAnnIds(imgIds=[image_id], iscrowd=None)
# anns = coco.loadAnns(ann_ids)

# # Convert PIL image to numpy for drawing
# img_np = np.array(img)

# # Create a blank mask of same size
# mask = np.zeros((img_np.shape[0], img_np.shape[1]), dtype=np.uint8)

# # Draw segmentations on mask
# for ann in anns:
#     segm = ann['segmentation']
#     if isinstance(segm, dict):
#         # If counts is a list, convert it to RLE
#         if isinstance(segm['counts'], list):
#             rle = maskUtils.frPyObjects(segm, img_info['height'], img_info['width'])
#             m = maskUtils.decode(rle)
#         else:
#             m = maskUtils.decode(segm)
#         mask = np.maximum(mask, m)



# # Overlay the mask on the image
# color_mask = np.zeros_like(img_np)
# color_mask[:, :, 1] = mask * 255  # Green mask

# # Blend image and mask
# alpha = 0.5
# img_with_mask = img_np.copy()
# img_with_mask = img_with_mask * (1 - alpha) + color_mask * alpha
# img_with_mask = img_with_mask.astype(np.uint8)

# # Show the result
# plt.figure(figsize=(10,10))
# plt.imshow(img_with_mask)
# plt.axis('off')
# plt.show()


In [ ]:
# import os
# import json
# import shutil
# import random
# from pycocotools.coco import COCO
# from PIL import Image

# # Paths
# COCO_ROOT = '/kaggle/input/coco-2017-dataset/coco2017'
# ANNOTATIONS_PATH = os.path.join(COCO_ROOT, 'annotations')
# TRAIN_IMAGES_PATH = os.path.join(COCO_ROOT, 'train2017')
# VAL_IMAGES_PATH = os.path.join(COCO_ROOT, 'val2017')

# WORK_DIR = '/kaggle/working/coco_person'
# TRAIN_DIR = os.path.join(WORK_DIR, 'train2017')
# VAL_DIR = os.path.join(WORK_DIR, 'val2017')
# TRAIN_LABELS_DIR = os.path.join(WORK_DIR, 'labels/train2017')
# VAL_LABELS_DIR = os.path.join(WORK_DIR, 'labels/val2017')

# os.makedirs(TRAIN_DIR, exist_ok=True)
# os.makedirs(VAL_DIR, exist_ok=True)
# os.makedirs(TRAIN_LABELS_DIR, exist_ok=True)
# os.makedirs(VAL_LABELS_DIR, exist_ok=True)

# # Load COCO datasets
# coco_train = COCO(os.path.join(ANNOTATIONS_PATH, 'instances_train2017.json'))
# coco_val = COCO(os.path.join(ANNOTATIONS_PATH, 'instances_val2017.json'))

# PERSON_CAT_ID = 1  # COCO person category_id

# def convert_polygon_to_yolo_segmentation(polygon, width, height):
#     """
#     Convert polygon segmentation coordinates to YOLO format normalized [x1,y1,...]
#     polygon: list of floats (x,y,x,y,...)
#     """
#     norm_seg = []
#     for i, coord in enumerate(polygon):
#         if i % 2 == 0:  # x coord
#             norm_seg.append(coord / width)
#         else:  # y coord
#             norm_seg.append(coord / height)
#     return norm_seg

# def process_coco_split(coco, images_dir, labels_dir, max_images=None):
#     img_ids = coco.getImgIds(catIds=[PERSON_CAT_ID])
#     random.shuffle(img_ids)
#     selected_img_ids = []

#     count = 0
#     for img_id in img_ids:
#         img_info = coco.loadImgs([img_id])[0]
#         anns = coco.loadAnns(coco.getAnnIds(imgIds=[img_id], catIds=[PERSON_CAT_ID], iscrowd=False))

#         # Filter annotations with valid polygon segmentations
#         valid_anns = []
#         for ann in anns:
#             seg = ann.get('segmentation', [])
#             if not seg or not isinstance(seg, list):
#                 continue
#             # Only polygon (list of list)
#             if isinstance(seg[0], list) and len(seg[0]) >= 6:  # at least 3 points
#                 valid_anns.append(ann)
#             elif isinstance(seg[0], float) and len(seg) >= 6:
#                 valid_anns.append(ann)

#         if len(valid_anns) == 0:
#             continue

#         # Copy image
#         src_img_path = os.path.join(images_dir, img_info['file_name'])
#         dst_img_path = os.path.join(images_dir.replace(COCO_ROOT, WORK_DIR), img_info['file_name'])
#         os.makedirs(os.path.dirname(dst_img_path), exist_ok=True)
#         shutil.copy2(src_img_path, dst_img_path)

#         # Write label file
#         label_file_path = os.path.join(labels_dir, img_info['file_name'].replace('.jpg', '.txt'))
#         lines = []
#         for ann in valid_anns:
#             polygons = ann['segmentation']
#             if isinstance(polygons[0], list):
#                 # multiple polygons - flatten all
#                 for poly in polygons:
#                     if len(poly) < 6:
#                         continue
#                     norm_seg = convert_polygon_to_yolo_segmentation(poly, img_info['width'], img_info['height'])
#                     line = "0 " + " ".join(f"{x:.6f}" for x in norm_seg)
#                     lines.append(line)
#             else:
#                 # single polygon
#                 if len(polygons) < 6:
#                     continue
#                 norm_seg = convert_polygon_to_yolo_segmentation(polygons, img_info['width'], img_info['height'])
#                 line = "0 " + " ".join(f"{x:.6f}" for x in norm_seg)
#                 lines.append(line)

#         if lines:
#             os.makedirs(os.path.dirname(label_file_path), exist_ok=True)
#             with open(label_file_path, 'w') as f:
#                 f.write("\n".join(lines))

#             count += 1
#             selected_img_ids.append(img_id)
#             if max_images and count >= max_images:
#                 break
#     return selected_img_ids, count

# print("Processing train set...")
# train_selected_ids, train_count = process_coco_split(
#     coco_train, TRAIN_IMAGES_PATH, TRAIN_LABELS_DIR, max_images=100
# )
# print(f"Copied {train_count} train images and labels.")

# print("Processing val set...")
# val_selected_ids, val_count = process_coco_split(
#     coco_val, VAL_IMAGES_PATH, VAL_LABELS_DIR, max_images=50
# )
# print(f"Copied {val_count} val images and labels.")

# # Create dataset config yaml for YOLOv8
# yaml_path = os.path.join(WORK_DIR, 'coco_person.yaml')
# with open(yaml_path, 'w') as f:
#     f.write(f"""
# names: ['person']
# nc: 1
# train: {TRAIN_DIR}
# val: {VAL_DIR}
# """)

# print(f"Dataset config saved to {yaml_path}")

# # Now train YOLOv8-seg on this dataset
# from ultralytics import YOLO

# model = YOLO('yolov8n-seg.pt')

# print("Starting training...")
# results = model.train(
#     data=yaml_path,
#     epochs=10,
#     imgsz=640,
#     batch=8,
#     device=0,
#     save=True,
#     name="yolov8n-seg-person"
# )

# print("Training done.")

# # You can then run evaluation or inference on validation images as needed.


In [ ]:
# !rm -rf /kaggle/working/d